# Gated Recurrent Units (GRUs)

A **Gated Recurrent Unit (GRU)** is a recurrent neural-network architecture that uses learned gates to control how information is carried from one time step to the next. This gated carry path helps mitigate the vanishing-gradient problem and makes long-range dependencies easier to learn than with a vanilla RNN.

> **Important terminology:** A standard GRU has one recurrent state, $h_t$. This vector is both the hidden state exposed as output and the memory passed to the next time step. A GRU does **not** have a separate cell-state vector $c_t$ like an LSTM. The phrase **GRU cell** means the entire computational unit—not a separate memory variable.

By the end of this notebook, you should understand the reset gate, update gate, candidate state, final hidden-state update, and the difference between GRU and LSTM memory.

## 1. Why use a GRU?

A vanilla RNN repeatedly transforms its previous hidden state:

$$
h_t = \tanh(W_xx_t + W_hh_{t-1} + b).
$$

Across a long sequence, repeatedly multiplying gradients by recurrent weights and activation derivatives can make them shrink. This is the **vanishing-gradient problem**, which makes distant dependencies difficult to learn. Gradients can also grow and become unstable—the **exploding-gradient problem**.

A GRU introduces paths that can carry the previous state forward with relatively little modification. This **mitigates** vanishing gradients; it does not guarantee that they disappear. GRUs also do not inherently eliminate exploding gradients, for which gradient clipping is commonly used.

## 2. Notation and shapes

At time step $t$:

| Symbol | Meaning | Shape for one example |
|---|---|---|
| $x_t$ | Current input | $(n_x,)$ |
| $h_{t-1}$ | Previous hidden state / previous memory | $(n_h,)$ |
| $r_t$ | Reset gate | $(n_h,)$ |
| $z_t$ | Update gate | $(n_h,)$ |
| $\tilde h_t$ | Candidate hidden state | $(n_h,)$ |
| $h_t$ | New hidden state / new memory | $(n_h,)$ |

Here, $n_x$ is the number of input features and $n_h$ is the hidden-state size. The gates are **vectors**, not single switches: each hidden-state component can be controlled by a different value. Their sigmoid outputs are continuous values between 0 and 1.

### Trainable parameter dimensions

Each of the reset gate, update gate, and candidate calculation needs an input-to-hidden matrix, a hidden-to-hidden matrix, and a bias vector:

| Calculation | Input weights | Shape | Recurrent weights | Shape | Bias | Shape |
|---|---|---|---|---|---|---|
| Reset gate $r_t$ | $W_r$ | $(n_h, n_x)$ | $U_r$ | $(n_h, n_h)$ | $b_r$ | $(n_h,)$ |
| Update gate $z_t$ | $W_z$ | $(n_h, n_x)$ | $U_z$ | $(n_h, n_h)$ | $b_z$ | $(n_h,)$ |
| Candidate $\tilde h_t$ | $W_h$ | $(n_h, n_x)$ | $U_h$ | $(n_h, n_h)$ | $b_h$ | $(n_h,)$ |

The $W$ matrices transform the current input from $n_x$ features to $n_h$ hidden features:

$$
W_r x_t:\quad (n_h, n_x)(n_x,) \longrightarrow (n_h,).
$$

The $U$ matrices transform the previous hidden state while preserving it at size $n_h$:

$$
U_r h_{t-1}:\quad (n_h, n_h)(n_h,) \longrightarrow (n_h,).
$$

Therefore, all terms in a gate equation have shape $(n_h,)$ and can be added. The same reasoning applies to the update-gate and candidate equations. For a batch of $B$ examples, the vectors are commonly stored as $(B,n_x)$ and $(B,n_h)$; libraries may multiply them by transposed versions of the matrices shown above.

Ignoring implementation-specific bias duplication, these three calculations contain

$$
3\big(n_hn_x+n_h^2+n_h\big)
$$

trainable scalar parameters in total.

## 3. GRU computation, step by step

### Step 1: Reset gate

$$
r_t = \sigma(W_rx_t + U_rh_{t-1} + b_r)
$$

The reset gate controls how much of the previous state participates in constructing the **candidate** state. If a component of $r_t$ is near 0, the corresponding component of $h_{t-1}$ contributes little to the candidate. If it is near 1, that previous-state component remains available while constructing the candidate.

> The reset gate does not directly erase the final hidden state. Even when $r_t$ is small, the update gate can still copy information from $h_{t-1}$ into $h_t$.

### Step 2: Candidate hidden state

$$
\tilde h_t = \tanh\left(W_hx_t + U_h(r_t \odot h_{t-1}) + b_h\right)
$$

$\tilde h_t$ is a proposal for the new state. The symbol $\odot$ denotes element-wise multiplication. The $\tanh$ activation bounds each candidate component between $-1$ and $1$.

### Step 3: Update gate

$$
z_t = \sigma(W_zx_t + U_zh_{t-1} + b_z)
$$

### Step 4: Final hidden state

$$
h_t = (1-z_t) \odot \tilde h_t + z_t \odot h_{t-1}.
$$

Under this notebook's convention:

- $z_t \approx 0$: favor the new candidate $\tilde h_t$.
- $z_t \approx 1$: retain the previous state $h_{t-1}$.

Thus, although $z_t$ is conventionally called the **update gate**, it directly weights the amount of old state retained in this equation; $1-z_t$ weights the amount updated. The calculation is an element-wise interpolation, so different state components can retain or replace information independently.

### A warning about update-gate conventions

Some books and implementations instead write

$$
h_t = z_t \odot \tilde h_t + (1-z_t) \odot h_{t-1}.
$$

In that convention, $z_t \approx 1$ means **update to the candidate**. The two formulations simply define the gate in complementary ways. When comparing sources, inspect the final-state equation rather than relying only on the name “update gate.”

## 4. Small numerical example

Suppose one component of the previous state is $h_{t-1}=0.8$, the corresponding candidate is $\tilde h_t=-0.2$, and the update-gate value is $z_t=0.9$. Then

$$
h_t=(1-0.9)(-0.2)+0.9(0.8)=0.70.
$$

Because $z_t$ is high, that component remains close to the old value. If $z_t=0.1$ instead,

$$
h_t=(1-0.1)(-0.2)+0.1(0.8)=-0.10,
$$

so the component moves much closer to the candidate. In a real GRU, this calculation happens simultaneously for every component of the hidden-state vector.

In [ ]:
import torch

torch.manual_seed(0)

batch_size = 3
sequence_length = 5
input_size = 4
hidden_size = 6

gru = torch.nn.GRU(
    input_size=input_size,
    hidden_size=hidden_size,
    batch_first=True,
)

x = torch.randn(batch_size, sequence_length, input_size)
output, h_n = gru(x)

print("input shape:       ", x.shape)
print("all output states: ", output.shape)
print("final hidden state:", h_n.shape)
print("Last output equals final state:", torch.allclose(output[:, -1], h_n[0]))

For this one-layer, one-directional GRU:

- `output[:, t, :]` contains $h_t$ for every item in the batch.
- `h_n[0]` contains the final state for every item.
- There is no separate `c_n`. By contrast, PyTorch's LSTM returns both `h_n` and `c_n`.

## 5. Why the carry path helps

When $z_t$ is close to 1, the update equation contains a nearly direct path from $h_{t-1}$ to $h_t$:

$$
h_t \approx h_{t-1}.
$$

Information—and gradient signal during backpropagation—can travel along this path without being completely transformed by a new $\tanh$ operation at every step. This makes long-term retention easier than in a vanilla RNN. The gates are learned jointly with the rest of the model through backpropagation; they are not manually programmed.

## 6. GRU versus LSTM

| Feature | GRU | LSTM |
|---|---|---|
| Recurrent state | One state, $h_t$ | Two states, $h_t$ and $c_t$ |
| Role of $h_t$ | Hidden state, output, and memory | Exposed/output hidden state |
| Role of $c_t$ | No separate $c_t$ | Internal cell state used for longer-term storage |
| Gates | Reset and update | Input, forget, and output |
| Parameters at equal hidden size | Usually fewer | Usually more |
| Typical computation | Usually lower | Usually higher |
| Empirical performance | Task-dependent; neither architecture is universally better | Task-dependent; neither architecture is universally better |

For an LSTM, the exposed hidden state is computed from its separate cell state:

$$
h_t=o_t\odot\tanh(c_t),
$$

where $o_t$ is the output gate. A GRU merges these roles into the single state $h_t$. For the same hidden size, a GRU therefore usually has fewer parameters and gate computations, although actual training speed depends on software and hardware.

## 7. Summary

- A GRU carries one learned state vector, $h_t$; this is its hidden state and memory.
- A **GRU cell** is the whole update unit. It does not imply a separate $c_t$.
- The reset gate $r_t$ controls how the old state contributes to the candidate $\tilde h_t$.
- The update gate $z_t$ blends the old state and candidate component by component.
- Gated carry paths mitigate vanishing gradients and help learn long-range dependencies.
- Unlike a GRU, an LSTM carries both a hidden state $h_t$ and a separate cell state $c_t$.